# Running a distance-3 surface code memory experiment on IQM Garnet

IQM Garnet, the quantum computer we will be using today, has 20 qubits laid out as follows:

<img src="Garnet_layout.png" alt="Layout of IQM Garnet" width="500"/>

Our aim is to use 17 of those 20 to build a distance-3 surface code. The qubits for this experiment are laid out as follows:

<img src="surfaceCode_layout.jpg" alt="Layout of surface code experiment qubits" width="500"/>

where data qubits are in blue, ancilla qubits for Z-type stabilizers are in orange, and ancilla qubits for X-type stabilizers are in purple.

# Preparing your environment

Let's install the packages we need:

In [ ]:
%%capture
!pip install "iqm-client[qrisp]"
!pip install matplotlib
!pip install pymatching

# Measuring a stabilizer

The core of the experiment is measuring the stabilizers. For example, consider the stabilizer which is the product of Pauli $Z$ operators acting on the four data qubits shown below:

<img src="qubit_layout.jpg" alt="Layout for four data qubits and one ancilla" width="250"/>

Note that we have just numbered the data qubits 0-3; this numbering does not refer to a specific stabilizer in the above layout! We're just assuming that we have any ancilla and its neighboring 4 qubits.

We could find out the value of this stabilizer by measuring all four data qubits, and multiplying the results. But that isn't what we want - we need to measure the stabilizer without measuring each qubit individually. We do this by entangling the ancilla qubit in the middle with all of the data qubits in the stabilizer.

The procedure we have in mind is the following:

1) Initialize the ancilla qubit to $\left|0\right\rangle$.
2) Using several 2-qubit controlled gates, flip the ancilla qubit if the stabilizer we are trying to measure is $-1$, and leave the ancilla qubit unchanged otherwise.
3) Measure the ancilla qubit.

Right now, we just want to implement step 2 - the initialization and measurement of the ancilla qubit will happen when we build the full error correction circuit. We will often still call step 2 "measuring the stabilizer," though.

**TASK**: Construct quantum circuits that measure the $Z$- and $X$-type stabilizers. (Start with $Z$; it is a little more intuitive.)

In [ ]:
from qrisp import *

def measure_z_stabilizer(data_qubits,ancilla_qubit):
    """
    Arguments: data_qubits: QuantumVariable (expected to be 2 or 4 qubits)
            ancilla_qubit: QuantumVariable (one qubit)
    Result: entangles ancilla qubit with data qubits so that measuring the ancilla qubit measures the Z-type
            stabilizer
    TASK: Add the necessary gates
    """
    pass

def measure_x_stabilizer(data_qubits,ancilla_qubit):
    """
    Inputs: data_qubits: QuantumVariable (expected to be 2 or 4 qubits)
            ancilla_qubit: QuantumVariable (one qubit)
    Result: entangles ancilla with data qubits so that measuring the ancilla qubit measures the X-type
            stabilizer
    TASK: Add the necessary gates
    """
    pass

ancilla1 = QuantumVariable(1)
ancilla2 = QuantumVariable(1)
data = QuantumVariable(4)
# Optionally insert some gates here to prepare different input states to test your function

measure_z_stabilizer(data,ancilla1)
print(ancilla1)

measure_x_stabilizer(data,ancilla2)
print(ancilla2)

IQM's native 2-qubit gates (the ones that are implemented in hardware and are therefore the most efficient gates to apply) are `CZ` gates, not `CX` gates. This means that our circuit will be more efficient if we replace any 2-qubit gates with `CZ` and single-qubit gates.

**TASK**: Rewrite the above circuits in terms of single-qubit gates and `CZ` gates, with no other 2-qubit gates. HINT: Hadamard gates generally interchange the roles of $X$ and $Z$. But on which qubit(s) should we apply a Hadamard gate?

In [ ]:
def measure_z_stabilizer(data_qubits,ancilla_qubit):
    """
    Inputs: data_qubits: QuantumVariable (expected to be 2 or 4 qubits)
            ancilla_qubit: QuantumVariable (one qubit)
    Result: entangles ancilla with data qubits so that measuring the ancilla qubit measures the Z-type
            stabilizer.
    TASK: Add the necessary gates. Use only single-qubit and CZ gates.
    """
    pass

def measure_x_stabilizer(data_qubits,ancilla_qubit):
    """
    Inputs: data_qubits: QuantumVariable (expected to be 2 or 4 qubits)
            ancilla_qubit: QuantumVariable (one qubit)
    Result: entangles ancilla with data qubits so that measuring the ancilla qubit measures the X-type
            stabilizer
    TASK: Add the necessary gates. Use only single-qubit and CZ gates.
    """
    pass


ancilla1 = QuantumVariable(1)
data = QuantumVariable(4)
# Insert some gates here to prepare different input states to test your function
x(data[0])
measure_z_stabilizer(data,ancilla1)
print(ancilla1)

ancilla2 = QuantumVariable(1)

h(data)
measure_x_stabilizer(data,ancilla2)
print(ancilla2)

**TASK**: Now write a function to measure all of the stabilizers in the distance-3 surface code, given that the qubits are laid out as above.

In [ ]:
def measure_all_stabilizers(data_qubits,ancilla_qubits):
    """
    Input: data_qubits: QuantumVariable (expected to be 9 qubits)
           ancilla_qubits: QuantumVariable (expected to be 8 qubits)
    Result: Entangle all the ancilla qubits with the data qubits so that measuring the ancilla qubits
              measures all the stabilizers in the distance-3 surface code
    TASK: write this function
    """

    pass


Now we run the circuit in a simulation.

In [ ]:
data = QuantumVariable(9,name="data_qb")
ancilla = QuantumVariable(8,name="anc_qb")

measure_all_stabilizers(data,ancilla)

print(data.qs)

syndromes=measure(ancilla)

res = data.qs.run(shots=1000)
res

**TASK:** Which ancillas sometimes come out 1 in simulation? Why?

# Initialization

In order to initialize the circuit, we need to
1) Initialize the data qubits into an eigenstate of the logical $Z$ string operator, i.e., to have either an even (logical 0) or odd (logical 1) number of qubits in the state $\left|1\right\rangle$ in each row.
2) Force the data qubits to all be in eigenstates of all of the stabilizers.

Conveniently, measuring a stabilizer actually forces the data qubits to be in an eigenstate of that stabilizer. So we just have to measure all the stabilizers in order to accomplish step 2.

How do we accomplish step 1?

Happily, if our data qubits start in the state $\left|000 \cdots 0\right\rangle$, they already have an even number of $\left|1\right\rangle$ in each row. Measuring the stabilizers cannot change this fact. So if we just proceed directly to step 2, after measuring the stabilizers, we will initialize into the logical $\left|0\right\rangle_L$ state (well, there are some subtleties here - we'll get to that soon).

But in order to initialize into a logical $\left|1\right\rangle_L$ state, we have to force the data qubits to have an **odd** number of $\left|1\right\rangle$ qubits in each row. Your job is to figure out how to do this.

**TASK: Write a function initializing the data qubits into the desired logical state.**

In [ ]:
def initialize_logical_state(desired_logical):
    """
    Input: desired_logical: String (expected to be either '0' or '1')
    Output: data_qubits: QuantumVariable(9) - data qubits, initialized into logical state |desired_logical>
            ancilla_qubits: QuantumVariable(8) - ancilla qubits for use in measuring stabilizers
    Note: if you call the measure() function on any qubits, the measurement results will automatically
          be output when you run the quantum circuit. You do not need to explicitly *return* any
          measurement results.
    """
    data_qubits = QuantumVariable(9)
    ancilla_qubits = QuantumVariable(8)
    if desired_logical == '0':
        pass
    elif desired_logical == '1':
        #apply a logical X operator
        [x(data_qubits[i]) for i in [2,5,8]] # one choice of logical X
    else:
        raise Exception("Invalid input for logical state")
    measure_all_stabilizers(data_qubits,ancilla_qubits)
    measure(ancilla_qubits)
    return data_qubits, ancilla_qubits

data, ancilla = initialize_logical_state('0') # You can try this with either input state

# Look at the circuit
print(data.qs)

This initialization procedure doesn't actually guarantee that we initialize into the logical states $\left|0\right\rangle_L$ or $\left|1\right\rangle_L$ though... To see why, let's see what syndromes we get when we test our function and measure the syndromes:

In [ ]:
test_syndromes = ancilla.qs.run(shots=1000)

test_syndromes

As you have noticed above, no matter what logical state we prepare, there is a pretty large chance that our initial syndromes will not all be 0. This means we are not actually in a logical state.

**TASK:** Some of the initial syndromes are always zero. Which ones? Why? Why aren't the other syndromes always zero?

This is not actually a problem; we just need to keep track of those initial syndromes as we decode. They will affect how we interpret the rest of the syndromes when we actually do an error correction layer. This is called tracking the "Pauli frame." Essentially, we're treating those initial syndromes like an initial set of errors and decoding them.

# The Error Correction Circuit

We now have all the tools we need for error correction.

**TASK: Define a function implementing one round of error correction.**

In [ ]:
def QEC_round(data_qubits, ancilla_qubits):
    """
    Input: data_qubits: QuantumVariable (expected to be 9 qubits)
           ancilla_qubits: QuantumVariable (expected to be 8 qubits)
    Result: Apply one round of error correction, including syndrome measurements.
    Note: As with initialization, you do not have to explicitly *return* any measurement results.
    """
    pass

# Putting it all together

Now you should write a function that implements the memory experiment circuit for a fixed number `num_rounds` of error correction: initialization, `num_rounds` of error correction, and data readout.

One important point: It would be ideal if we could reset the ancilla qubits in between rounds of error correction, so that every time you see an ancilla as 1, it would mean that there is an error. **Please do not reset the ancilla qubits in your circuit!**

Why? IQM hardware does have a reset operation! However, it is slow enough that a noticeable amount of noise will be introduced to our circuit. In order to get the best performance, we will not reset the ancilla qubits.

In [ ]:
def memory_experiment(logical_state,num_rounds):
    """
    Inputs: logical_state: str (expected to be '0' or '1')
            num_rounds: int (number of rounds of error correction)
    Output: data_qubits: QuantumVariable(9)
            ancilla_qubits: QuantumVariable(8)
    Function should initialize data_qubits into the logical state logical_state, perform num_rounds of
            error correction, then measure the data qubits.
    TASK: finish this function
    """
    pass

def parse_result(measurements):
    """
    Function to parse a full set of measurement results into a list of data readout and syndromes
    Input: string of measurement outputs.
      Note: by default, measurement results in qrisp appear in *reverse* order, that is, the last measurement done in the circuit is the first
      measurement that appears in the measurement result string. So for us, the measurement results we will be parsing is the concatenation of
      data_readout, syndromes_round_n, syndromes_round_nMinusOne, ..., syndromes_round_1, syndromes_initialization.
    Output: list of strings [syndromes_initialization, syndromes_round_1, syndromes_round_2,..., syndromes_round_n, data_readout]
    """
    bitstring = measurements[::-1]
    rounds=(len(bitstring)-9)//8
    output = [bitstring[(8*i):(8*(i+1))] for i in range(rounds)] # break string into syndromes, ignoring data qubits
    output.append(bitstring[-9:]) # Add the data qubit results to the end
    return output

num_rounds = 4
simulated_data, simulated_ancilla = memory_experiment('0',num_rounds)
res_sim=simulated_data.qs.run(shots=100)
# res is a dictionary where the measurement results are listed in reverse order.

for result in res_sim.keys():
    print(parse_result(result))

# Running the experiment on real hardware

Now we want to connect to IQM Garnet and run the experiment. Our goal is to run the experiment for many different numbers of rounds of QEC. This is pretty straightforward!

In [ ]:
from iqm.qrisp_iqm import IQMBackend
import os

# Turn on automatic dynamical decoupling in order to mitigate some errors
from iqm.iqm_client.models import CircuitCompilationOptions
compOptions = CircuitCompilationOptions(dd_mode='enabled')

os.environ["IQM_SERVER_URL"] = "https://resonance.iqm.tech/"
os.environ["IQM_TOKEN"] = input("Input your IQM Resonance API token")

quantum_computer = IQMBackend(device_instance = "garnet",
                         compilation_options= compOptions)

num_rounds_max = 6
num_shots = 1000

initial_state = '0'

result_list = []

for rounds in range(num_rounds_max+1):
    real_QPU_data,real_QPU_ancilla = memory_experiment(initial_state,rounds)
    res=real_QPU_data.qs.run(shots=num_shots, backend=quantum_computer)
    result_list.append(res)

for result in result_list[0].keys():
    print(parse_result(result))

# Decoding with PyMatching

Above, we built the memory experiment from scratch so that you could see the nuts and bolts of what is going on in the experiment. The package `Pymatching` greatly simplifies decoding by implementing an algorithm called Minimum Weight Perfect Matching (MWPM).

`PyMatching` requires us to build the so-called "matching graph." We will not discuss the details of this today; just run the following cell to prepare the function that builds the matching graph.

In [ ]:
import numpy as np
import pymatching

N_DATA = 9

# Each entry lists the data-qubit indices in that Z stabilizer.
Z_STABILIZERS = [
    [0, 1, 3, 4],   # A1 (bulk)
    [2, 5],   #  A3 (right boundary)
    [3,6],   # A4 (left boundary)
    [4,5,7,8]         # A6 (bulk)
]

X_STABILIZERS = [
    [0, 1],   # A0 (bottom boundary)
    [1, 2, 4, 5],   # A2 (bulk)
    [3, 4, 6, 7],         # A5 (bulk)
    [7, 8],         # A7 (top boundary)
]

N_STAB = len(Z_STABILIZERS)

# Representative of the logical Z operator (bottom row).
LOGICAL_Z_SUPPORT = [0, 1, 2]


def _build_data_qubit_edges():
    """
    For each data qubit, find which Z stabilizer(s) it belongs to.
    Returns a list of (qubit, stab_a, stab_b_or_None) tuples:
      - stab_b is None  -> qubit only affects one Z stabilizer (a boundary edge)
      - stab_b is an int -> qubit affects exactly two Z stabilizers (an internal edge)
    """
    membership = {q: [] for q in range(N_DATA)}
    for stab_idx, stab in enumerate(Z_STABILIZERS):
        for q in stab:
            membership[q].append(stab_idx)

    edges = []
    for q in range(N_DATA):
        stabs = membership[q]
        if len(stabs) == 1:
            edges.append((q, stabs[0], None))
        elif len(stabs) == 2:
            edges.append((q, stabs[0], stabs[1]))
        else:
            raise ValueError(
                f"Data qubit {q} touches {len(stabs)} Z stabilizers; "
                "expected 1 (boundary) or 2 (internal)."
            )
    return edges


DATA_QUBIT_EDGES = _build_data_qubit_edges()


# ---------------------------------------------------------------------------
# Matching graph construction
# ---------------------------------------------------------------------------

def build_matching_graph(n_rounds, p_data=1e-3, p_meas=1e-3):
    """
    Build the space-time PyMatching graph for n_rounds of Z-stabilizer
    measurement followed by one virtual round derived from the final
    data-qubit measurement.

    Node (stab_idx, layer) for layer = 0 .. n_rounds  (n_rounds+1 layers total,
    the last one being the virtual "data-derived" layer).

    Edges:
      - one copy of each data-qubit edge per layer ("space-like" edges,
        representing an X error occurring during that time step). Edges
        touching the logical-Z support (qubits 0,1,2) carry fault_id 0.
      - time-like edges between consecutive REAL rounds only (there is no
        independent measurement-error edge into the virtual final layer,
        since its own noise is already captured by that layer's space-like
        edges).

    Returns the pymatching.Matching object plus a helper for turning
    (stab_idx, layer) into a node id.
    """
    if n_rounds < 1:
        raise ValueError("n_rounds must be >= 1")

    n_layers = n_rounds + 1

    def node_id(stab_idx, layer):
        return layer * N_STAB + stab_idx

    w_data = np.log((1 - p_data) / p_data)
    w_meas = np.log((1 - p_meas) / p_meas)

    m = pymatching.Matching()

    # Space-like (data-qubit-error) edges, one independent copy per layer.
    for layer in range(n_layers):
        for (q, a, b) in DATA_QUBIT_EDGES:
            fault_ids = {0} if q in LOGICAL_Z_SUPPORT else set()
            na = node_id(a, layer)
            if b is None:
                m.add_boundary_edge(
                    na, fault_ids=fault_ids, weight=w_data,
                    error_probability=p_data, merge_strategy="independent",
                )
            else:
                nb = node_id(b, layer)
                m.add_edge(
                    na, nb, fault_ids=fault_ids, weight=w_data,
                    error_probability=p_data, merge_strategy="independent",
                )

    # Time-like (measurement-error) edges, between consecutive real rounds.
    for layer in range(n_rounds - 1):
        for si in range(N_STAB):
            m.add_edge(
                node_id(si, layer), node_id(si, layer + 1),
                fault_ids=set(), weight=w_meas,
                error_probability=p_meas, merge_strategy="independent",
            )

    return m, node_id, n_layers


## Bookkeeping

The next cell defines some functions for parsing the results. They are tedious "bookkeeping" exercises.

If you want to understand why they are here:

1.   Our measurement is preparing a logical $\left|0\right\rangle$ or $\left|1\right\rangle$ state. It turns out that we only need to decode the Z-type syndromes for this experiment, since phase errors do not directly cause Z logical errors. So the function `remove_x_syndromes` throws out the X-type syndromes. (It also reverses the measurement results, since they come in from the machine "backwards".)
2.   The function `bitstring_to_arrays` just reshapes the measurements into convenient shapes for PyMatching.
3.  The function `compute_detection_events` figures out whether a syndrome has detected a *new* error in each round. Roughly speaking, this asks if the syndrome changed state. There's some extra bookkeeping because we don't reset our ancillas.



In [ ]:
def remove_x_syndromes(counts):
    counts_no_x_syndromes = {}
    for bitstring, shots in counts.items():
        num_rounds=(len(bitstring)-9)//8
        reversed_bitstring = bitstring[::-1]
        short_bitstring = ''
        z_stab_positions = [1,3,4,6] #which ancilla qubit numbers represent Z stabilizers
        for i in reversed(range(num_rounds)):
            for j in z_stab_positions:
                short_bitstring += reversed_bitstring[8*i+j]
        short_bitstring += reversed_bitstring[-9:]  # Add the data qubit results to the end
        if short_bitstring not in counts_no_x_syndromes:
            counts_no_x_syndromes[short_bitstring] = shots
        else:
            counts_no_x_syndromes[short_bitstring] += shots
    return counts_no_x_syndromes

def bitstring_to_arrays(bitstring, n_rounds):
    """
    Parse a bitstring of length 4*n_rounds + 9 into:
      rounds: shape (n_rounds, 4) array of measured Z-stabilizer outcomes
      data:   shape (9,) array of final data-qubit Z measurement outcomes
    """
    bitstring = bitstring.strip()
    expected_len = n_rounds * N_STAB + N_DATA
    if len(bitstring) != expected_len:
        raise ValueError(
            f"Expected bitstring length {expected_len} "
            f"(= {n_rounds} rounds * {N_STAB} + {N_DATA} data bits), "
            f"got {len(bitstring)} for '{bitstring}'."
        )
    bits = np.array([int(c) for c in bitstring], dtype=np.uint8)
    rounds = bits[: n_rounds * N_STAB].reshape(n_rounds, N_STAB)
    data = bits[n_rounds * N_STAB:]
    return rounds, data

def compute_detection_events(rounds, data, n_rounds, n_layers, node_id):
    """
    Turn raw (rounds, data) measurement outcomes into a detector vector
    suitable for pymatching.Matching.decode(), ASSUMING ANCILLAS ARE NOT
    RESET between stabilizer-measurement rounds.

    Without reset, the CNOT sequence for round t doesn't start from a clean
    |0> ancilla -- it XORs the current stabilizer parity s_t onto whatever
    classical value m_{t-1} was left over from the previous round's
    measurement. So the raw measured bit is a running cumulative XOR:

        m_t = m_{t-1} XOR s_t      (m_{-1} := 0)

    We first "undo" this accumulation to recover the instantaneous per-round
    stabilizer value s_t = m_t XOR m_{t-1}. Once recovered, the rest of the
    pipeline (differencing consecutive layers, folding in the data-derived
    final layer) is identical to the ancilla-reset case.
    """
    recovered = np.zeros_like(rounds)
    prev = np.zeros(N_STAB, dtype=np.uint8)
    for t in range(n_rounds):
        recovered[t] = rounds[t] ^ prev
        prev = rounds[t]

    final_layer = np.array(
        [int(np.bitwise_xor.reduce(data[stab])) for stab in Z_STABILIZERS],
        dtype=np.uint8,
    )
    layers = np.vstack([recovered, final_layer[None, :]])  # (n_layers, N_STAB)

    det = np.zeros_like(layers)
    det[0] = layers[0]
    for t in range(1, n_layers):
        det[t] = layers[t] ^ layers[t - 1]

    det_vec = np.zeros(N_STAB * n_layers, dtype=np.uint8)
    for layer in range(n_layers):
        for si in range(N_STAB):
            det_vec[node_id(si, layer)] = det[layer, si]
    return det_vec


## Actually decoding
Finally, we can actually decode!

In [ ]:
def compute_logical_error_rate(
    counts,
    n_rounds,
    initial_logical_value=0,
):
    """
    Decode every bitstring in `counts` and return the logical error rate.

    Parameters
    ----------
    counts : dict[str, int]
        Keys are bitstrings of length 4*n_rounds + 9 (see module docstring
        for the assumed bit ordering). Values are observed counts.
    n_rounds : int
        Number of syndrome-extraction rounds encoded in each bitstring.
    initial_logical_value : int (0 or 1)
        The known logical Z eigenvalue of the prepared initial state.

    Returns
    -------
    float
        errors / total_shots, i.e. the fraction of shots for which the
        decoder's predicted correction disagrees with the actual logical
        flip inferred from the raw data (a logical error / decoding
        failure).
    """
    matching, node_id, n_layers = build_matching_graph(n_rounds)

    total = 0
    errors = 0

    for bitstring, count in counts.items():
        # Preprocess the data as discussed above
        rounds, data = bitstring_to_arrays(bitstring, n_rounds)
        det_vec = compute_detection_events(rounds, data, n_rounds, n_layers, node_id)

        # Run the actual decoder! This tells you whether or not the decoder thinks there was a logical error
        prediction = matching.decode(det_vec)
        predicted_flip = int(np.atleast_1d(prediction)[0])

        # Was there an actual logical bit-flip?
        measured_logical = int(np.bitwise_xor.reduce(data[LOGICAL_Z_SUPPORT]))
        actual_flip = measured_logical ^ initial_logical_value

        total += count
        if (actual_flip ^ predicted_flip) != 0:
            errors += count

    return errors / total if total > 0 else float("nan")

Let's decode, and plot the results!

In [ ]:
ler_list = [compute_logical_error_rate(remove_x_syndromes(result_list[r]),r+1) for r in range(len(result_list))]

import matplotlib.pyplot as plt
plt.plot(ler_list, 'o')
plt.xlabel('QEC round')
plt.ylabel('Logical error rate')
plt.xticks(range(num_rounds+2))
plt.axis([-0.1,num_rounds+0.9,0,0.6])
plt.show()

These results aren't very good - we hit 50% error rate (random noise) pretty quickly. But we've done essentially zero optimization - good QEC requires a lot of optimization in the circuit construction, and further down the stack as well!

# Reminder of the full experiment

Our aim is to plot the logical error rate as a function of the number of rounds of QEC rounds. There are two reasons this is useful.

First, we can imagine that between each round of QEC, we are doing a step of some calculation. The goal of this memory experiment is to find out how much logical error gets introduced in each step, which is obtained by fitting the data in the aforementioned plot. Ideally, this would be lower than the error rate if we didn't do QEC and just worked with the physical qubits. We won't be able to squeeze that performance out of Garnet just yet (if we did, it would be a publishable result!), but we can see how far we have to go.

Second, if we had more qubits, we could calculate the logical error rate per QEC round for different code distances. If this number decreases as the code distance gets bigger, then we are said to be "below threshold." This is a huge milestone for a quantum computers. More physical qubits usually means more errors. But if we're below threshold, then adding more physical qubits lets us suppress the *logical* error rate as much as we like, even though those physical qubits do introduce more *physical* errors. This means our QEC scheme is useful!

# Conclusion

You have just run your first QEC experiment on quantum hardware! You created the QEC circuit, ran it on IQM Garnet, decoded with PyMatching, and analyzed the results.

In this notebook, we have done a huge amount of this by hand. There are many pre-built, highly efficient libraries that package up large amounts of the circuit construction and analysis for you, particularly if you want to run simulations (including simulations with noise, which we haven't done in this notebook). But there is still a lot of work to be done - as of this writing, we are still in the relatively early stages of running QEC experiments on quantum hardware. We hope that this workshop gives you the tools to start running your own QEC experiments.

In [ ]:

# Copyright 2026 IQM Quantum Computers (Daniel Bulmash)
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.